# Article Recommendation System

The goal is to recommend, for an article that a reader is reading, other articles of the same website that deal with a similar subject (content-based recommendation). The data has 34 articles of a data science blog, each with its text and its title (`articles.csv`).

Reference solution: [Article Recommendation System with Machine Learning](https://amanxai.com/2021/11/10/article-recommendation-system-with-machine-learning/) (Aman Kharwal). Dataset: [articles.csv](https://raw.githubusercontent.com/amankharwal/Website-data/master/articles.csv).

The reference turns the article texts into TF-IDF vectors, computes the cosine similarity between all pairs and writes the titles of four recommended articles next to each article. It checks the result by looking at one article. This notebook looks at the details of that code, and measures the quality of the recommendations with topics assigned to the articles by hand.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import os
os.chdir('/mnt/c/Users/myama/OneDrive/Belgeler/ai-1/Assignments/14-Specialize in Data Science/Recommendation Systems/article_recommendation_system')

## Data Loading

In [2]:
data = pd.read_csv("articles.csv", encoding="latin1")
print(data.shape)
data["Title"].head(10)

(34, 2)


0                    Best Books to Learn Data Analysis
1           Assumptions of Machine Learning Algorithms
2            News Classification with Machine Learning
3    Multiclass Classification Algorithms in Machin...
4          Multinomial Naive Bayes in Machine Learning
5            News Classification with Machine Learning
6                              Best Books to Learn NLP
7                 Send Instagram Messages using Python
8       Pfizer Vaccine Sentiment Analysis using Python
9           Squid Game Sentiment Analysis using Python
Name: Title, dtype: str

## Reference Reproduction

The reference code is reproduced first, on the data as it is. It passes the list of articles as the `input` parameter of `TfidfVectorizer`, which only accepts `'content'`, `'filename'` or `'file'`; older versions of scikit-learn ignored the mistake and the current one raises an error, so the parameter is left out. `argsort()[-5:-1]` takes the four articles with the highest similarity except the very last one, which is supposed to be the article itself (its similarity to itself is 1). This works only if the article is alone in the first place: two articles with identical text both have a similarity of 1 to each other, and then the last position can be taken by the copy instead of the article itself.

In [3]:
articles = data["Article"].tolist()

try:
    text.TfidfVectorizer(input=articles, stop_words="english").fit_transform(articles)
except Exception as error:
    print("the reference call fails with this scikit-learn version:", type(error).__name__)

uni_matrix = text.TfidfVectorizer(stop_words="english").fit_transform(articles)
uni_sim = cosine_similarity(uni_matrix)

def recommend_articles(x):
    return ", ".join(data["Title"].loc[x.argsort()[-5:-1]])

data["Recommended Articles"] = [recommend_articles(x) for x in uni_sim]
print(data["Title"][22], "->", data["Recommended Articles"][22])
print("articles that recommend themselves:", [i for i, (t, r) in enumerate(zip(data["Title"], data["Recommended Articles"])) if t in r.split(", ")])

the reference call fails with this scikit-learn version: InvalidParameterError
Agglomerative Clustering in Machine Learning -> BIRCH Clustering in Machine Learning, Clustering Algorithms in Machine Learning, DBSCAN Clustering in Machine Learning, K-Means Clustering in Machine Learning
articles that recommend themselves: [2, 5]


## Data Cleaning

One article is in the file twice with the same text ("News Classification with Machine Learning"), which produces the problem above and would recommend a copy of the article to itself. One character in the texts is not readable (`\x8b`, an encoding artifact). The duplicate is removed and the character is replaced by a space.

In [4]:
print("duplicated rows:", data.duplicated(subset=["Article", "Title"]).sum())
data = data.drop(columns="Recommended Articles").drop_duplicates(subset=["Article", "Title"]).reset_index(drop=True)
data["Article"] = data["Article"].str.replace(r"[^\x00-\x7f]", " ", regex=True)
data["words"] = data["Article"].str.split().str.len()
data["words"].describe().round(0)

duplicated rows: 1


count     33.0
mean      77.0
std       19.0
min       38.0
25%       66.0
50%       80.0
75%       94.0
max      111.0
Name: words, dtype: float64

Each article has only 38 to 111 words (77 on average), an excerpt and not the full text. Few words per document means that a similarity based on words has little to work with.

## Feature Engineering

The recommendations cannot be judged without knowing which articles belong together, so each article is assigned by hand to a topic from its title. Only 33 articles exist, so the topics are few and small. Articles with a topic of their own (a single article) are left out of the evaluation.

In [5]:
topic_rules = {
    "clustering": "Clustering|K-Means",
    "naive bayes": "Naive Bayes",
    "learning resources": "Best Books|Best Resources",
    "neural networks": "Neural Network|Perceptron|Applications of Deep Learning",
    "sentiment analysis": "Sentiment",
    "stock prices": "Stock Price",
    "python code": "Python Dictionary|Python List|Send Instagram|Voice Recorder|Scatter Plot|Python Frameworks",
    "classification": "Classification|Language Detection",
    "machine learning overview": "Assumptions of Machine|Use Cases of Different",
}
data["topic"] = "single"
for topic, pattern in topic_rules.items():
    data.loc[data["Title"].str.contains(pattern), "topic"] = topic

data.loc[data["topic"] == "single", "topic"] = "single " + data.index[data["topic"] == "single"].astype(str)
data["topic"].str.replace(r"single \d+", "single article", regex=True).value_counts()

topic
python code                  6
clustering                   6
learning resources           5
classification               3
neural networks              3
machine learning overview    2
naive bayes                  2
sentiment analysis           2
single article               2
stock prices                 2
Name: count, dtype: int64

## Model Training

The reference vectorization is compared with a few alternatives: the same TF-IDF on the title and the text together, sequences of one to two words, and simple word counts (`CountVectorizer`) instead of TF-IDF. All use the cosine similarity. A random ranking is the baseline. The recommendations are the four most similar articles, best first (the reference lists them from the least to the most similar).

In [6]:
texts = {"text": data["Article"], "title and text": data["Title"] + ". " + data["Article"]}
vectorizers = {
    "TF-IDF, text (reference)": (text.TfidfVectorizer(stop_words="english"), "text"),
    "TF-IDF, title and text": (text.TfidfVectorizer(stop_words="english"), "title and text"),
    "TF-IDF, words and pairs, text": (text.TfidfVectorizer(stop_words="english", ngram_range=(1, 2)), "text"),
    "Word counts, text": (CountVectorizer(stop_words="english"), "text"),
}
similarities = {name: cosine_similarity(vectorizer.fit_transform(texts[source])) for name, (vectorizer, source) in vectorizers.items()}
similarities["Random"] = np.random.default_rng(42).random((len(data), len(data)))

## Evaluation

In [8]:
topics = data["topic"].to_numpy()
same_topic = topics[:, None] == topics[None, :]
np.fill_diagonal(same_topic, False)
group_size = same_topic.sum(axis=1)
evaluated = group_size > 0

def scores(similarity, k=3):
    similarity = similarity.copy()
    np.fill_diagonal(similarity, -np.inf)
    top = np.argsort(-similarity, axis=1)[:, :k]
    hits = np.take_along_axis(same_topic, top, axis=1)
    return {"top-1 in the same topic": hits[evaluated, 0].mean(),
            "top-3 in the same topic (normalised)": (hits.sum(axis=1)[evaluated] / np.minimum(k, group_size[evaluated])).mean()}

print("evaluated articles:", evaluated.sum(), "of", len(data))
pd.DataFrame({name: scores(similarity) for name, similarity in similarities.items()}).T.round(3)

evaluated articles: 31 of 33


,top-1 in the same topic,top-3 in the same topic (normalised)
"TF-IDF, text (reference)",0.774,0.704
"TF-IDF, title and text",0.742,0.737
"TF-IDF, words and pairs, text",0.742,0.715
"Word counts, text",0.677,0.731
Random,0.032,0.108


In [9]:
similarity = similarities["TF-IDF, title and text"]
similarity_no_self = similarity.copy()
np.fill_diagonal(similarity_no_self, -np.inf)

for title in ["Agglomerative Clustering in Machine Learning", "Squid Game Sentiment Analysis using Python", "Best Books to Learn NLP", "Swap Items of a Python List"]:
    i = data.index[data["Title"] == title][0]
    print(title, "->", data["Title"].iloc[np.argsort(-similarity_no_self[i])[:3]].tolist())

Agglomerative Clustering in Machine Learning -> ['K-Means Clustering in Machine Learning', 'DBSCAN Clustering in Machine Learning', 'Clustering Algorithms in Machine Learning']
Squid Game Sentiment Analysis using Python -> ['Pfizer Vaccine Sentiment Analysis using Python', 'Best Books to Learn Data Analysis', 'Best Resources to Learn Python']
Best Books to Learn NLP -> ['Best Books to Learn Deep Learning', 'Best Books to Learn Computer Vision', 'Best Books to Learn Data Analysis']
Swap Items of a Python List -> ['For Loop Over Keys and Values in a Python Dictionary', 'Best Resources to Learn Python', 'Clustering Algorithms in Machine Learning']


The reference code needs three corrections to run and to give sensible lists: the `input` parameter must be removed (it raises an error in the current scikit-learn), the duplicated article must be removed (the two copies of "News Classification with Machine Learning" recommend themselves), and the four recommendations should be listed from the most to the least similar (the reference lists the least similar of the four first).

All the text-based variants are far better than a random ranking: the most similar article belongs to the same topic for 68% to 77% of the articles (3% at random). With only 31 evaluated articles, one article changes a score by 3.2 points, so the differences between the variants (74% and 77% for the first recommendation, 70% to 74% for the three first) are within one or two articles and cannot be called real. The reference TF-IDF on the text is as good as the other variants on this data.

The examples show the typical behaviour. Topics with a clear vocabulary (clustering algorithms, book lists) are found easily. The sentiment analysis article of the Squid Game series gets its sibling about the Pfizer vaccine first, but the next two suggestions are unrelated learning resources. "Swap Items of a Python List" gets the article about Python dictionaries first, then generic learning material: the texts of these code snippets are so short and generic that their words say little about their subject.

## Conclusion

**Reference approach.** The reference vectorizes the article texts with TF-IDF, computes the cosine similarity between all pairs, and writes four recommended titles next to each article with `argsort()[-5:-1]`. It checks one article by eye ("Agglomerative Clustering", whose four recommendations are all clustering articles).

**Findings.** The reference call `TfidfVectorizer(input=articles)` fails in the current scikit-learn (`InvalidParameterError`). The trick of dropping the last position of the sorted scores only removes the article itself when its similarity is unique: because the file contains one article twice, both copies of "News Classification with Machine Learning" recommend themselves. The four titles are also listed from the least to the most similar. The texts are excerpts of 38 to 111 words.

**Our approach.** After the corrections (duplicate removed, non-readable character replaced, ranking from the best), the quality was measured with topics assigned by hand from the titles (31 articles in topics of at least two articles), for the first and for the three first recommendations:

| Method | Top-1 in the same topic | Top-3 in the same topic (normalised) |
|---|---|---|
| TF-IDF, text (reference) | 77.4% | 70.4% |
| TF-IDF, title and text | 74.2% | 73.7% |
| TF-IDF, words and pairs, text | 74.2% | 71.5% |
| Word counts, text | 67.7% | 73.1% |
| Random | 3.2% | 10.8% |

Content-based recommendations are far better than random ones, and the variants cannot be told apart with 31 articles.

**Limitations.** The dataset is very small (33 articles), so every score is coarse and the topics, assigned by hand from the titles, are a judgement. The evaluation only checks whether a recommended article has the same topic, not whether a reader would find it useful. Short and generic texts (code snippets) give weak recommendations. A recommender for a real website would need the full texts and reader behaviour (clicks, reading time), which this data does not have.